In [1]:
!pip install pyarrow==2 awswrangler

In [2]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
import awswrangler as wr

In [3]:
# if import error run this cell first
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [4]:
import gspread
import boto3
import pandas as pd

gc = gspread.service_account(
    filename="/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/quick-cogency-314404-6a4762e0bc8d.json"
)

s3 = boto3.resource("s3")

In [37]:
# Fetch data from Google sheet
# NOTE: Change SERVICE ACCOUNT
gc = gspread.service_account(
    "/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/quick-cogency-314404-6a4762e0bc8d.json"
)


# retail marketing spend  budget
# https://docs.google.com/spreadsheets/d/1ygDhDd9rWv7R4aU8W7NkRMVTJ-N_bmi9JFg6ttqBkug/edit?ts=6070315b#gid=1436820854
sh = gc.open_by_key("1ygDhDd9rWv7R4aU8W7NkRMVTJ-N_bmi9JFg6ttqBkug")
wk = sh.worksheet("Data science")
retail_marketing_budget = pd.DataFrame.from_dict(wk.get_all_records())

In [38]:
retail_marketing_budget = retail_marketing_budget.drop(
    columns=["May", "June", "July", "August"]
)

In [39]:
retail_marketing_budget

,Branch,Abbreviation,September,October,November,December
0,All Seasons-TH,ALS,0.00,0.00,0.00,0.00
1,Central Festival Phuket-TH,CFP,199.66,1239.47,233.97,373.59
2,Central Lardprao-TH,CLP,592.12,669.65,283.37,541.69
3,Central Pinklao-TH,CPK,419.90,575.38,316.99,437.06
4,Central Rama 3-TH,CR3,380.80,1109.59,277.88,464.50
5,Central Rama 9-TH,CR9,491.26,1237.21,354.04,475.82
6,CentralWorld-TH,CTW,809.62,1746.31,1118.37,876.17
7,Emquartier-TH,EMQ,980.46,1968.75,1254.91,1002.42
8,Fashion Island-TH,FSI,325.22,299.83,188.00,364.33
9,Iconsiam-TH,ICS,1777.73,2644.30,1709.12,1145.47


In [40]:
# store mapping
# https://docs.google.com/spreadsheets/d/1TTM285yQXTxK_I0Kzb3MYq0XftBI6Wjf7WaBNq9traw/edit#gid=0
sh = gc.open_by_key("1TTM285yQXTxK_I0Kzb3MYq0XftBI6Wjf7WaBNq9traw")
wk = sh.worksheet("Sheet1")
store_mapping = pd.DataFrame.from_dict(wk.get_all_records())

In [41]:
store_mapping

,id_shop,erply_store_id,ns_store_id,store_name_db,people_counter_name,store_name_0119_0421,store_name_from_0421
0,1,11,14,Icon Siam,IconSiam,Iconsiam,ICS
1,1,211,29,Mega Bangna,MEGABANGNA,MegaBangna,MBN
2,1,221,31,Central Pinklao,Central Pinklao,Central Pinklao,CPK
3,1,231,69,All Seasons Place,CRC,All Seasons,ALS
4,1,241,70,Interchange 21,Interchange21,Interchange,
5,1,251,71,EmQuartier,Emquartier,Emquartier,EMQ
6,1,261,72,Central World,Central World,CentralWorld,CTW
7,1,281,73,Silom Complex,Silom,Silom Complex,Silom
8,1,291,96,Werk Ari,Ari,Ari,Ari
9,1,301,99,Central Phuket,Phuket,Central Festival Phuket,CFP


In [42]:
sql = """
select distinct month_id
                        ,month_name
            from dwh.dim_calendar
            order by month_id

"""

In [43]:
monthid = wr.athena.read_sql_query(sql, database="dwh")

In [44]:
monthid

,month_id,month_name
0,1,January
1,2,February
2,3,March
3,4,April
4,5,May
5,6,June
6,7,July
7,8,August
8,9,Septembe
9,10,October


In [45]:
monthid["month_name"] = monthid["month_name"].str.strip()
monthid["month_name"] = monthid["month_name"].replace("Septembe", "September")

In [46]:
monthid

,month_id,month_name
0,1,January
1,2,February
2,3,March
3,4,April
4,5,May
5,6,June
6,7,July
7,8,August
8,9,September
9,10,October


In [47]:
retail_budget_unpivot = retail_marketing_budget.melt(
    id_vars=["Branch", "Abbreviation"],
    var_name="month_name",
    value_name="retail_mkt_spend",
)
retail_budget_unpivot = retail_budget_unpivot.dropna()
retail_budget_unpivot.rename({"Branch": "store_name"}, axis=1, inplace=True)
retail_budget_unpivot.rename(
    {"Abbreviation": "store_name_from_0421"}, axis=1, inplace=True
)

retail_budget_unpivot_1 = retail_budget_unpivot.merge(
    store_mapping[["id_shop", "erply_store_id", "store_name_from_0421"]],
    how="left",
    on="store_name_from_0421",
)
retail_budget_unpivot_1 = retail_budget_unpivot_1.dropna()
convert = {"id_shop": int, "erply_store_id": int}
retail_budget_unpivot_1 = retail_budget_unpivot_1.astype(convert)

retail_budget_unpivot_1["year"] = 2021
retail_budget_unpivot_1 = retail_budget_unpivot_1.merge(
    monthid, how="left", on="month_name"
)
convert = {"month_id": int}
retail_budget_unpivot_1 = retail_budget_unpivot_1.astype(convert)
retail_budget_unpivot_1.rename({"erply_store_id": "store_id"}, axis=1, inplace=True)

In [48]:
def year_month_id(df):
    if df["month_id"] < 10:
        return str(df["year"]) + "0" + str(df["month_id"])
    elif df["month_id"] >= 10:
        return str(df["year"]) + str(df["month_id"])


retail_budget_unpivot_1["year_month_id"] = retail_budget_unpivot_1.apply(
    year_month_id, axis=1
)
retail_budget_unpivot_1.rename({"store_name": "store"}, axis=1, inplace=True)

In [49]:
retail_budget_unpivot_1

,store,store_name_from_0421,month_name,retail_mkt_spend,id_shop,store_id,year,month_id,year_month_id
0,All Seasons-TH,ALS,September,0.00,1,231,2021,9,202109
1,Central Festival Phuket-TH,CFP,September,199.66,1,301,2021,9,202109
2,Central Lardprao-TH,CLP,September,592.12,1,341,2021,9,202109
3,Central Pinklao-TH,CPK,September,419.90,1,221,2021,9,202109
4,Central Rama 3-TH,CR3,September,380.80,1,321,2021,9,202109
...,...,...,...,...,...,...,...,...,...
131,POP UP-Central Eastville-TH,,December,0.00,1,391,2021,12,202112
132,Paragon-TH,Paragon,December,89.54,1,391,2021,12,202112
133,POP UP-Siam Premium Outlet-TH,,December,0.00,1,241,2021,12,202112
134,POP UP-Siam Premium Outlet-TH,,December,0.00,1,391,2021,12,202112


In [50]:
convert = {"year_month_id": int, "store_id": int, "id_shop": int}
retail_budget_unpivot_1 = retail_budget_unpivot_1.astype(convert)

In [51]:
retail_budget_unpivot_1[
    [
        "year",
        "month_id",
        "year_month_id",
        "store",
        "store_id",
        "retail_mkt_spend",
        "month_name",
        "id_shop",
    ]
]

,year,month_id,year_month_id,store,store_id,retail_mkt_spend,month_name,id_shop
0,2021,9,202109,All Seasons-TH,231,0.00,September,1
1,2021,9,202109,Central Festival Phuket-TH,301,199.66,September,1
2,2021,9,202109,Central Lardprao-TH,341,592.12,September,1
3,2021,9,202109,Central Pinklao-TH,221,419.90,September,1
4,2021,9,202109,Central Rama 3-TH,321,380.80,September,1
...,...,...,...,...,...,...,...,...
131,2021,12,202112,POP UP-Central Eastville-TH,391,0.00,December,1
132,2021,12,202112,Paragon-TH,391,89.54,December,1
133,2021,12,202112,POP UP-Siam Premium Outlet-TH,241,0.00,December,1
134,2021,12,202112,POP UP-Siam Premium Outlet-TH,391,0.00,December,1


In [52]:
retail_marketing_spend = retail_budget_unpivot_1[
    [
        "year",
        "month_id",
        "year_month_id",
        "store",
        "store_id",
        "retail_mkt_spend",
        "month_name",
        "id_shop",
    ]
]

In [56]:
retail_marketing_spend["retail_mkt_spend"] = retail_marketing_spend[
    "retail_mkt_spend"
].astype(int)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':


In [57]:
retail_marketing_spend = retail_marketing_spend[
    retail_marketing_spend["retail_mkt_spend"] > 0
]

In [58]:
retail_marketing_spend

,year,month_id,year_month_id,store,store_id,retail_mkt_spend,month_name,id_shop
1,2021,9,202109,Central Festival Phuket-TH,301,199,September,1
2,2021,9,202109,Central Lardprao-TH,341,592,September,1
3,2021,9,202109,Central Pinklao-TH,221,419,September,1
4,2021,9,202109,Central Rama 3-TH,321,380,September,1
5,2021,9,202109,Central Rama 9-TH,311,491,September,1
...,...,...,...,...,...,...,...,...
126,2021,12,202112,Jem-SG,42,914,December,2
127,2021,12,202112,Silom Complex-TH,281,6,December,1
129,2021,12,202112,Kota Kasablanca-ID,35,302,December,5
132,2021,12,202112,Paragon-TH,391,89,December,1


In [59]:
# have to join on year, month, sc, id_shop_name

In [60]:
import gspread
import boto3

gc = gspread.service_account(
    filename="/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/quick-cogency-314404-6a4762e0bc8d.json"
)

s3 = boto3.resource("s3")

In [61]:
# q1 2022
# https://docs.google.com/spreadsheets/d/1SrVHNjf3Ozs_plKFK5itdirqlIh7BKa72UPv9i5v5c0/edit#gid=0
sh = gc.open_by_key("1SrVHNjf3Ozs_plKFK5itdirqlIh7BKa72UPv9i5v5c0")
wk = sh.worksheet("Sheet1")
retail_marketing_q12022 = pd.DataFrame.from_dict(wk.get_all_records())

In [62]:
retail_marketing_q12022

,year,month_name,month_id,year_month_id,store,store_id,country,id_shop,retail_mkt_spend,oldcluster,newcluster
0,2022,January,1,202201,Terminal 21,361,TH,1,481.43,sc3,FFF
1,2022,January,1,202201,Fashion Island,381,TH,1,986.88,sc3,FFF
2,2022,January,1,202201,Central Pinklao,221,TH,1,1017.92,sc3,FFF
3,2022,January,1,202201,Central Park,15,ID,5,1267.73,sc3,FFF
4,2022,January,1,202201,Kota Kasablanca,35,ID,5,1490.29,sc3,FFF
...,...,...,...,...,...,...,...,...,...,...,...
56,2022,March,3,202203,CentralWorld,261,TH,1,1990.59,sc2,FF
57,2022,March,3,202203,Iconsiam,11,TH,1,2338.75,sc2,PF
58,2022,March,3,202203,MegaBangna,211,TH,1,2508.66,sc1,FF
59,2022,March,3,202203,Siam Center,371,TH,1,2774.83,sc2,FF


In [63]:
retail_marketing_spend = retail_marketing_spend.merge(
    retail_marketing_q12022[["store_id", "id_shop", "oldcluster", "newcluster"]],
    how="left",
    on=["store_id", "id_shop"],
)

In [64]:
retail_marketing_spend

,year,month_id,year_month_id,store,store_id,retail_mkt_spend,month_name,id_shop,oldcluster,newcluster
0,2021,9,202109,Central Festival Phuket-TH,301,199,September,1,sc3,LN
1,2021,9,202109,Central Festival Phuket-TH,301,199,September,1,sc3,LN
2,2021,9,202109,Central Festival Phuket-TH,301,199,September,1,sc3,LN
3,2021,9,202109,Central Lardprao-TH,341,592,September,1,sc1,FF
4,2021,9,202109,Central Lardprao-TH,341,592,September,1,sc1,FF
...,...,...,...,...,...,...,...,...,...,...
251,2021,12,202112,Kota Kasablanca-ID,35,302,December,5,sc3,FFF
252,2021,12,202112,Kota Kasablanca-ID,35,302,December,5,sc3,FFF
253,2021,12,202112,Kota Kasablanca-ID,35,302,December,5,sc3,FFF
254,2021,12,202112,Paragon-TH,391,89,December,1,NaN,NaN


In [65]:
# 2021

retail_marketing_spend2021 = (
    retail_marketing_spend.groupby(["year", "month_name", "id_shop", "oldcluster"])[
        "retail_mkt_spend"
    ]
    .mean()
    .reset_index()
)

In [66]:
# 2022

retail_marketing_spend2022 = (
    retail_marketing_q12022.groupby(["year", "month_name", "id_shop", "oldcluster"])[
        "retail_mkt_spend"
    ]
    .mean()
    .reset_index()
)

In [67]:
retail_marketing_spend = pd.concat(
    [retail_marketing_spend2021, retail_marketing_spend2022]
)

In [68]:
retail_marketing_spend.rename(columns={"oldcluster": "store_cluster"}, inplace=True)

In [69]:
retail_marketing_spend.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_retail_marketing_spend.csv",
    index=False,
)